# Demographics & KPI EDA

This notebook explores customer demographic patterns across the A/B experiment groups and analyzes how demographic variables such as age, gender, and tenure relate to key behavioral KPIs.

The analysis supports the Tableau dashboard and final business interpretation by identifying customer behavior trends, engagement patterns, and potential differences between Test and Control groups.

In [ ]:
# Import required libraries

import pandas as pd
import numpy as np

## Load Data

We start with the final client-level KPI table because it contains the client demographics, experiment group assignment, and client-level KPI metrics in one table.

We also load the cleaned demographics table only as a sanity check source.

In [ ]:
# Load final client-level KPI table
df = pd.read_csv("../data/clean/client_kpi_table.csv")

# Optional sanity-check source: cleaned demographics table
df_demo = pd.read_csv("../data/clean/df_demo_clean.csv")

In [ ]:
# Check shape of both datasets

print("Client KPI table shape:", df.shape)
print("Clean demographics table shape:", df_demo.shape)

In [ ]:
# Check available columns in the client KPI table

df.columns

In [ ]:
# Preview the client KPI table

df.head()

## Initial Dataset Exploration & Quality Checks

We first explore the structure and quality of the dataset before analyzing demographic patterns and KPI behavior across experiment groups.

Checks include:
- missing values
- duplicate client IDs
- experiment group distribution
- demographic completeness

In [ ]:
# Check missing values

df.isnull().sum()

In [ ]:
# Check for duplicate client IDs

df["client_id"].duplicated().sum()

In [ ]:
# Check Test vs Control distribution

df["Variation"].value_counts()

In [ ]:
# Percentage distribution of experiment groups

df["Variation"].value_counts(normalize=True) * 100

In [ ]:
# Missing values in key demographic columns

df[["clnt_age", "gendr", "clnt_tenure_mnth"]].isnull().sum()

In [ ]:
# Gender distribution

df["gendr"].value_counts(dropna=False)

### Initial observations

- The dataset is structured at client level, with one row per client.
- No duplicate client IDs were found.
- The Test and Control groups are well balanced in size, allowing reliable comparison between experiment groups.
- Key demographic variables (age, gender, tenure) contain no missing values.
- The gender column includes many "U" values, representing unknown or unspecified gender.
- The variable `avg_time_to_completion` contains many missing values because it is only available for clients who completed the process.

## Demographic Overview

We first explore the main demographic variables in the client-level KPI table.

This helps us understand the customer base before comparing demographic segments with KPI behavior.

In [ ]:
# Summary statistics for key demographic variables

df[["clnt_age", "clnt_tenure_mnth", "num_accts", "bal", "calls_6_mnth", "logons_6_mnth"]].describe()

## Create Demographic Segments

To make the demographic analysis easier to interpret in Tableau, we create grouped variables for age and tenure.

In [ ]:
# Create age groups for easier comparison

df["age_group"] = pd.cut(
    df["clnt_age"],
    bins=[0, 30, 40, 50, 60, 70, 120],
    labels=["<30", "30-39", "40-49", "50-59", "60-69", "70+"]
)

# Check age group distribution
df["age_group"].value_counts().sort_index()

### Demographic observations

- The client base is primarily middle-aged, with the largest groups between 30 and 60 years old.
- Clients aged 50–59 represent the largest age segment in the dataset.
- Very young clients (<30) and older clients (70+) are less represented.
- The average client age is approximately 47 years.
- Most clients have between 1 and 2 accounts.
- Client tenure is relatively high overall, suggesting that many users are long-term Vanguard customers.
- Digital engagement appears moderate to high, with clients averaging around 6 logins and 3 calls over 6 months.

**Methodological Note**  
This notebook uses aggregated client-level KPI metrics from `client_kpi_table` for demographic analysis purposes.  

The KPI definitions used here may slightly differ from the formally constructed frictionless completion and friction metrics developed in the hypothesis testing notebook, which were built directly from visit-level behavioral data (`visits_kpi_table` and `events_kpi_table`).  

The purpose of this notebook is exploratory demographic segmentation analysis rather than formal statistical hypothesis testing.

## Completion Rate by Age Group

We now explore whether completion behavior differs across age segments.

This helps identify whether certain demographic groups responded differently to the redesign experience.

In [ ]:
# Completion rate by age group

df.groupby("age_group")["completion_rate"].mean().sort_index()

### Completion rate observations by age group

- Completion rates are highest among clients between 30 and 39 years old.
- Clients under 50 generally show stronger completion behavior than older age groups.
- Completion rates decline progressively with age, especially after age 50.
- Clients aged 70+ have the lowest completion rate in the dataset.
- This may suggest that the redesigned experience is easier to complete for younger and middle-aged users compared to older clients.

## Completion Rate by Age Group and Experiment Variation

We now compare completion behavior across age groups separately for the Test and Control groups.

This helps evaluate whether certain demographic segments benefited more from the redesign.

In [ ]:
# Completion rate by age group and experiment variation

df.groupby(["age_group", "Variation"])["completion_rate"].mean().unstack()

### Completion rate observations by age group and experiment variation

- The Test variation shows slightly higher completion rates for most younger and middle-aged groups.
- Clients aged below 50 generally performed better in the redesigned experience.
- However, clients aged 50+ do not consistently benefit from the redesign.
- In the 50–59 and 70+ groups, the Control group slightly outperforms the Test group.
- This may suggest that the new user experience is more effective for younger users, while older clients may experience more friction in the redesigned process.

### Visualization: Completion Rate by Age Group and Experiment Variation

The following chart highlights how completion behavior differs across age groups for both the Test and Control experiences.

In [ ]:
import matplotlib.pyplot as plt

# Prepare data
completion_age_var = df.groupby(
    ["age_group", "Variation"]
)["completion_rate"].mean().unstack()

# Create chart
completion_age_var.plot(kind="bar", figsize=(8,5))

# Labels
plt.title("Completion Rate by Age Group and Experiment Variation")
plt.xlabel("Age Group")
plt.ylabel("Completion Rate")
plt.xticks(rotation=0)

plt.show()


## Error Rate by Age Group

We now explore whether certain age groups experience more process errors during the client journey.

This helps identify potential usability challenges across demographic segments.

In [ ]:
# Error rate by age group

df.groupby("age_group")["error_rate"].mean().sort_index()

### Error rate observations by age group

- Error rates increase progressively with age.
- Clients below 40 show the lowest proportion of process errors.
- Clients aged 60+ experience the highest error rates in the dataset.
- The 70+ segment has the highest error frequency overall.
- This pattern may indicate that older users face more usability difficulties during the process flow.
- The redesigned experience may require additional accessibility or simplification improvements for older client segments.

## Average Steps per Client by Age Group

We now analyze how many process steps clients complete on average across different age segments.

This helps evaluate whether some demographic groups require more navigation effort during the process.

In [ ]:
# Average steps per client by age group

df.groupby("age_group")["avg_steps_client"].mean().sort_index()

### Average steps observations by age group

- The average number of steps per client remains relatively stable across age groups.
- Clients aged 50–69 show the highest average number of process steps.
- This may suggest that middle-to-older age groups require slightly more navigation effort during the process.
- Interestingly, the 70+ group shows a lower average number of steps compared to the 50–69 segments.
- One possible explanation is that some older users abandon the process earlier instead of continuing through additional steps.

## Error Rate by Age Group and Experiment Variation

We now compare process error rates across age groups separately for the Test and Control variations.

This helps evaluate whether the redesigned experience affects demographic segments differently in terms of usability and process friction.

In [ ]:
# Error rate by age group and experiment variation

df.groupby(["age_group", "Variation"])["error_rate"].mean().unstack()

### Error rate observations by age group and experiment variation

- Across every age segment, the Test variation shows higher error rates than the Control variation.
- Error rates increase steadily with age in both experiment groups.
- The largest error rates are observed among clients aged 60+ and especially 70+.
- The redesigned experience appears to introduce additional process friction compared to the original flow.
- Older users seem to be disproportionately affected by the redesign, suggesting potential usability or accessibility challenges for senior client segments.

### Visualization: Error Rate by Age Group and Experiment Variation

The following chart compares process error rates across demographic age segments for both experiment variations.

In [ ]:
# Prepare data
error_age_var = df.groupby(
    ["age_group", "Variation"]
)["error_rate"].mean().unstack()

# Create chart
error_age_var.plot(kind="bar", figsize=(8,5))

# Labels
plt.title("Error Rate by Age Group and Experiment Variation")
plt.xlabel("Age Group")
plt.ylabel("Error Rate")
plt.xticks(rotation=0)

plt.show()

## Completion Rate by Gender and Experiment Variation

We now compare completion rates across gender categories for both experiment variations.

This helps evaluate whether the redesigned experience affects demographic groups differently.

In [ ]:
# Completion rate by gender and experiment variation

df.groupby(["gendr", "Variation"])["completion_rate"].mean().unstack()

### Completion rate observations by gender and experiment variation

- Completion rates are relatively similar across gender groups.
- Male clients show the highest completion rates overall.
- Female clients have the lowest completion rates among the available gender categories.
- The Test variation slightly improves completion rates for male and unknown-gender clients.
- For female clients, the redesigned experience shows a very small decrease in completion rate compared to the Control group.
- Overall, gender appears to have a weaker influence on completion behavior than age.

## Error Rate by Gender and Experiment Variation

We now compare process error rates across gender categories for both experiment variations.

This helps identify whether some demographic groups experience more friction or usability issues in the redesigned process.

In [ ]:
# Error rate by gender and experiment variation

df.groupby(["gendr", "Variation"])["error_rate"].mean().unstack()

### Error rate observations by gender and experiment variation

- Across all gender categories, the Test variation produces higher error rates than the Control variation.
- Female clients show the highest error rates overall.
- Male clients have slightly lower error rates compared to the other gender groups.
- The increase in error rates between Control and Test is consistent across all gender categories.
- This suggests that the redesigned experience may introduce additional usability friction regardless of gender.
- Compared to age, gender differences appear relatively moderate.

## Completion Rate by Client Tenure

We now analyze whether completion behavior differs across client tenure segments.

This helps evaluate whether long-term and newer clients respond differently to the redesigned experience.

In [ ]:
# Create tenure groups

df["tenure_group"] = pd.cut(
    df["clnt_tenure_mnth"],
    bins=[0, 60, 120, 180, 240, 1000],
    labels=["0-5y", "5-10y", "10-15y", "15-20y", "20y+"]
)

# Check tenure group distribution

df["tenure_group"].value_counts().sort_index()

## Completion Rate by Client Tenure Group

We now compare completion behavior across different client tenure segments.

This helps identify whether newer or long-term clients adapt differently to the redesigned experience.

In [ ]:
# Completion rate by tenure group

df.groupby("tenure_group")["completion_rate"].mean().sort_index()

### Key observations

- Completion rates are relatively similar across tenure groups overall.
- Clients with 5–10 years of tenure show the highest completion rates.
- Very long-term clients (20y+) show the lowest completion rates.
- This may suggest that highly established users experience more friction with the redesigned process.
- Differences across tenure groups appear smaller than the differences observed across age groups.
- Overall, age seems to have a stronger relationship with completion behavior than client tenure.

## Completion Rate by Tenure Group and Experiment Variation

We now compare completion behavior across tenure segments separately for the Test and Control groups.

This helps evaluate whether newer or long-term clients respond differently to the redesigned experience.

In [ ]:
# Completion rate by tenure group and experiment variation

df.groupby(["tenure_group", "Variation"])["completion_rate"].mean().unstack()

### Key observations

- Clients with lower and mid-level tenure (0–15 years) show slightly higher completion rates in the Test variation.
- The strongest improvement appears in the 5–10 year segment.
- For clients with 15+ years of tenure, the redesigned experience does not improve completion behavior.
- In the 20y+ segment, the Control group slightly outperforms the Test group.
- This may suggest that long-term clients are more attached to the old process and experience greater friction with the redesign.
- Overall, the redesigned experience seems more effective for newer and moderately established clients than for very long-term customers.

# Overall Insights

Based on the demographic and KPI analysis, several important behavioral patterns emerge across the experiment groups.

### Main findings

- Age appears to be the strongest demographic factor influencing client behavior.
- Younger clients generally achieve higher completion rates and lower error rates.
- Older clients, especially users aged 60+, experience substantially higher error rates and lower completion success.
- The redesigned Test experience appears to benefit younger users more consistently than older users.
- Gender differences exist but appear relatively small compared to age-related effects.
- Client tenure shows moderate influence on completion behavior, with very long-term clients responding less positively to the redesign.

### Business interpretation

- The redesigned experience may introduce additional usability friction for older and highly established clients.
- Long-term users may be more accustomed to the previous workflow and therefore less adaptable to interface changes.
- Younger and digitally engaged clients appear to navigate the redesigned process more successfully.

### Recommendations

- Consider simplifying navigation and reducing friction points for older demographic groups.
- Conduct deeper UX investigation into where older users encounter process errors.
- Explore whether personalized onboarding or guided flows could improve completion rates for long-term clients.
- Future analysis could investigate step-level drop-offs and interaction patterns in more detail.

### Tableau opportunities

This dataset is well suited for interactive Tableau dashboards including:
- Completion rate by demographic segment
- Error rate by age and experiment variation
- KPI comparison between Test and Control groups
- Demographic filtering and drill-down analysis